In [2]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [3]:
raw = Path("../datasets/asl_raw/asl_alphabet_train/asl_alphabet_train")

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(raw, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(dataset.classes)

print("Classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Classes: 29
Training images: 69600
Validation images: 17400


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

In [7]:
#Freezing the pretrained fetayre layers
for param in model.features.parameters():
    param.requires_grad = False

In [8]:
# Replace the final layer for ASL classification
model.classifier[6] = nn.Linear(4096, num_classes)

In [9]:
model = model.to(device)

print("Model ready on:", device)

Model ready on: cpu


In [10]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.01
)

In [ ]:
import time

epochs = 3

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    accuracy = 100 * correct / total
    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {accuracy:.2f}% "
        f"Time: {elapsed:.2f}s"
    )